We implement two NACE-agnostic baselines (3 static neighbourhoods recommended for every class) to compare with the FT-LLM/RAG systems of the initial proxy-framing:

- TopPop:     the three neighbourhoods with the most businesses
- Centrality: the three neighbourhoods closest to the city centre

Each is scored against the proxy top-3 (Match@3, Precision@3) over the same 159 inference queries used for the FT-LLM and RAG evaluations, so the numbers are directly comparable.

## **Import & Setup**

In [1]:
import json
import re
import numpy as np
import pandas as pd

In [2]:
inference_path  = "../../07. Approach - Fine Tuning/1. Fine-Tuning - Inference Datasets Creation/3. Fine-Tuning - Inference Data Creation/Extracted Files/inference_dataset.jsonl"
proxy_top3_path = "../../06. Ground Truths Creation/4. Ensembling of Approaches/Extracted CSV Files/top3_final.csv"

## **Help Functions**

In [6]:
# Function that returns Match@3 and Precision@3 over all queries.
def score_constant_baseline(constant_top3):
    match_flags = []
    precision_values = []
    predicted = set(str(name) for name in constant_top3)
    for nace_code in query_nace_codes:
        if nace_code not in proxy_top3_by_nace:
            continue
        ground_truth = set(str(name) for name in proxy_top3_by_nace[nace_code])
        overlap = len(ground_truth & predicted)
        match_flags.append(1 if overlap > 0 else 0)
        precision_values.append(overlap / 3)
    return float(np.mean(match_flags)), float(np.mean(precision_values))


## **Load the Data**

In [3]:
interaction_matrix    = pd.read_parquet("../data/interaction_matrix_raw.parquet")   # NACE x neighbourhood counts
neighbourhood_features = pd.read_parquet("../data/neighborhood_features.parquet")
proxy_top3_table       = pd.read_csv(proxy_top3_path)

neighbourhoods = list(interaction_matrix.columns)
neighbourhood_features = neighbourhood_features.loc[neighbourhoods]

# proxy top-3 per NACE class, keyed by the class code rounded to 4 decimals
proxy_top3_by_nace = {
    round(float(row["NACE Code"]), 4): [row["Top1"], row["Top2"], row["Top3"]]
    for _, row in proxy_top3_table.iterrows()
}

## **Create the 2 Baselines**

In [4]:
# Create the 2 static baselines:
business_count_per_neighbourhood = interaction_matrix.sum(axis=0)
toppop_top3     = list(business_count_per_neighbourhood.sort_values(ascending=False).head(3).index)
centrality_top3 = list(neighbourhood_features["distance_to_volos_center_km"].sort_values().head(3).index)

print("TopPop     top-3 (most businesses):     ", toppop_top3)
print("Centrality top-3 (closest to centre):   ", centrality_top3)

TopPop     top-3 (most businesses):      ['Agios Nikolaos', 'Metamorfosi', 'Nea Ionia']
Centrality top-3 (closest to centre):    ['Agios Vasilios', 'Analipsi', 'Metamorfosi']


## **Evaluate the 2 Baselines**

In [5]:
# Extract the NACE class of each inference query:
query_nace_codes = []
with open(inference_path) as inference_file:
    for line in inference_file:
        record = json.loads(line)
        nace_match = re.search(r"NACE\s+(\d+\.\d+)", record["completion"])
        if nace_match:
            query_nace_codes.append(round(float(nace_match.group(1)), 4))

covered = sum(code in proxy_top3_by_nace for code in query_nace_codes)
print(f"queries: {len(query_nace_codes)} | unique NACE classes: {len(set(query_nace_codes))} | covered by proxy: {covered}")

queries: 159 | unique NACE classes: 53 | covered by proxy: 159


In [8]:
# Evaluate the baselines
results = {
    "TopPop":      score_constant_baseline(toppop_top3),
    "Centrality":  score_constant_baseline(centrality_top3),
}
results

{'TopPop': (0.7358490566037735, 0.27672955974842767),
 'Centrality': (0.8490566037735849, 0.3899371069182391)}

In [9]:
# Compare the baselines to the LLM systems (using the documented LLM metrics values)
results["FT-LLM (paper)"] = (0.780, 0.319)
results["RAG (paper)"]    = (0.830, 0.367)

results_table = pd.DataFrame(
    [(name, round(match, 3), round(precision, 3)) for name, (match, precision) in results.items()],
    columns=["Method", "Match@3", "Precision@3 (=Recall@3)"],
).sort_values("Match@3", ascending=False)
print(results_table.to_string(index=False))

        Method  Match@3  Precision@3 (=Recall@3)
    Centrality    0.849                    0.390
   RAG (paper)    0.830                    0.367
FT-LLM (paper)    0.780                    0.319
        TopPop    0.736                    0.277
